<a href="https://colab.research.google.com/github/cathrineq/python-ai-Tarasova-Kate/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- [`currency.csv`](https://github.com/cathrineq/python-ai-Tarasova-Kate/blob/main/data/currency.csv) — данные о валютах из Wikidata: код, название, цена в евро, год, единица измерения

**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файл в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [5]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

repo = "python-ai-Tarasova-Kate"  # ← изменено: имя вашего репозитория
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Tarasova-Kate
    !git clone -q https://github.com/cathrineq/python-ai-Tarasova-Kate.git  # ← изменено: ваш URL репозитория

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Tarasova-Kate


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [6]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

df_currency = pd.read_csv("data/currency.csv")  # ← изменено: ваш файл с валютами

print("✅ Загружено строк в df_currency:", len(df_currency))  # ← изменено: имя переменной
print("✅ Столбцы:", df_currency.columns.tolist())  # ← добавлено: быстрая проверка структуры

✅ Загружено строк в df_currency: 2348
✅ Столбцы: ['currency', 'currencyLabel', 'shortName', 'price', 'year', 'unitLabel', 'unitSymbol']


## 🧹 [2B] Очистка и переименование столбцов

В вашем CSV-файле `currency.csv` есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `currency` с URL (ссылкой на объект Wikidata) — **удаляем его**, так как в данном анализе он не требуется.
- Столбцы `currencyLabel`, `unitLabel` содержат читаемые подписи (названия валют и единиц).

В этом шаге мы:
- удалим столбец с URL Wikidata (`currency`);
- переименуем `currencyLabel → currency`, `unitLabel → unit` (уберём постфикс `Label`);
- приведём числовые столбцы (`price`, `year`) к типу `float`/`int`.

При приведении к числам мы используем:

- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(int)` или `astype(float)` — переводит столбец к нужному числовому типу.

> ⚠️ **Важно:** если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа. Столбец с URL можно сохранить, если позже понадобится переход к оригинальной записи в Викиданных — но в этом уроке мы его удаляем для простоты.

In [11]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для currency.csv

# 1) Удаляем технический столбец с URL Wikidata, если он есть
if "currency" in df_currency.columns and df_currency["currency"].astype(str).str.contains("wikidata.org", na=False).any():
    df_currency = df_currency.drop(columns=["currency"])  # ← удаляем столбец с URL
    print("🗑️ Столбец 'currency' (URL) удалён")
else:
    print("⏭️ Столбец с URL не найден или уже удалён")

# 2) Переименовываем столбцы с постфиксом Label
df_currency = df_currency.rename(columns={
    "currencyLabel": "currency",  # ← иракский динар → теперь в столбце 'currency'
    "unitLabel": "unit"           # ← евро → теперь в столбце 'unit'
}, errors="ignore")               # ← игнорируем, если столбца уже нет
print("✅ Столбцы переименованы: currencyLabel → currency, unitLabel → unit")

# 3) Приводим числовые столбцы к правильным типам
# price → float (цены могут быть дробными)
df_currency["price"] = pd.to_numeric(
    df_currency["price"], errors="coerce"
).fillna(0).astype(float)

# year → int (год — целое число)
df_currency["year"] = pd.to_numeric(
    df_currency["year"], errors="coerce"
).fillna(0).astype(int)

print("✅ Числовые столбцы (price, year) приведены к нужным типам")
print("\n✅ Данные готовы к анализу")

# 🔍 Проверка: смотрим результат очистки
print("📋 Столбцы после очистки:", df_currency.columns.tolist())
print("\n📊 Первые 3 строки:")
display(df_currency.head(3))

⏭️ Столбец с URL не найден или уже удалён
✅ Столбцы переименованы: currencyLabel → currency, unitLabel → unit
✅ Числовые столбцы (price, year) приведены к нужным типам

✅ Данные готовы к анализу
📋 Столбцы после очистки: ['currency', 'shortName', 'price', 'year', 'unit', 'unitSymbol']

📊 Первые 3 строки:


,currency,shortName,price,year,unit,unitSymbol
0,иракский динар,NaN,0.00071,2020,евро,د.ع
1,иракский динар,NaN,0.00071,2020,евро,ID
2,кувейтский динар,NaN,3.28190,2016,доллар США,د.ك


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame `df_currency`:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по числовым столбцам (`price`, `year`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы быстро получать сводку по данным.

**Столбцы в нашем датасете:**
- `currency` — название валюты (например, «иракский динар»)
- `shortName` — короткий код/аббревиатура
- `price` — цена валюты в евро (числовой)
- `year` — год данных (числовой)
- `unit` — единица измерения (например, «евро»)
- `unitSymbol` — символ единицы (например, «د.ع»)

In [12]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов, первые строки и статистика."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))

    print("\nПервые строки:")
    print(df.head(n))

    # 🔢 Базовая статистика по числовым столбцам
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 0:
        print(f"\n📈 Статистика по числовым столбцам ({', '.join(numeric_cols)}):")
        print(df[numeric_cols].describe())
    else:
        print("\n⚠️ Нет числовых столбцов для статистики")

# 🔍 Шаг 3. Обзор данных
show_info(df_currency, "Курсы валют (df_currency)")


📊 Курсы валют (df_currency)
Размер: (2348, 6)
Столбцы: currency, shortName, price, year, unit, unitSymbol

Первые строки:
           currency shortName    price  year        unit unitSymbol
0    иракский динар       NaN  0.00071  2020        евро        د.ع
1    иракский динар       NaN  0.00071  2020        евро         ID
2  кувейтский динар       NaN  3.28190  2016  доллар США        د.ك
3  кувейтский динар       NaN  3.00000  2016  доллар США        د.ك
4  кувейтский динар       NaN  3.28190  2016  доллар США         KD

📈 Статистика по числовым столбцам (price, year):
              price         year
count  2.348000e+03  2348.000000
mean   5.549886e+05  1013.577087
std    2.094162e+07  1010.418892
min    0.000000e+00     0.000000
25%    0.000000e+00     0.000000
50%    5.000000e-01  1855.500000
75%    2.000000e+00  2025.000000
max    1.000000e+09  2026.000000


## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** валют, единиц измерения и символов есть в данных;
- **какие единицы измерения встречаются чаще всего** (Топ‑5 по числу строк);
- **какие валюты представлены в датасете** (список уникальных названий);
- **диапазон цен и лет**: минимальная/максимальная цена валюты в евро, период данных.

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому
`df_currency["unit"].value_counts().head()` даёт **Топ‑5 единиц измерения по числу записей**.

> 💡 **Подсказка:** если в данных есть дубликаты (одна и та же валюта с разными символами), `value_counts()` поможет их быстро обнаружить.

In [13]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных: валюты")

# 📊 Основные метрики: уникальные значения
print("\n📊 Уникальные значения:")
print("Уникальных валют:", df_currency["currency"].nunique())
print("Уникальных единиц измерения:", df_currency["unit"].nunique())
print("Уникальных символов:", df_currency["unitSymbol"].nunique())
if "shortName" in df_currency.columns:
    print("Уникальных коротких названий (shortName):", df_currency["shortName"].nunique())

# 🏆 Топ-5 единиц измерения по числу записей
print("\n🏆 Топ-5 единиц измерения (unit) по числу записей:")
print(df_currency["unit"].value_counts().head())

# 💱 Список уникальных валют
print("\n💱 Уникальные валюты в датасете:")
for curr in df_currency["currency"].unique():
    print(f"  • {curr}")

# 🔣 Распределение символов единиц (unitSymbol)
print("\n🔣 Символы единиц измерения (unitSymbol):")
print(df_currency["unitSymbol"].value_counts())

# 📈 Анализ числовых столбцов: price и year
print("\n📊 Числовые показатели:")
print(f"Диапазон цен (price): {df_currency['price'].min():.6f} — {df_currency['price'].max():.6f} евро")
print(f"Диапазон лет (year): {df_currency['year'].min()} — {df_currency['year'].max()}")

print("\n📈 Статистика по цене (price):")
print(df_currency["price"].describe())

print("\n📈 Статистика по году (year):")
print(df_currency["year"].describe())

🔍 Быстрая проверка данных: валюты

📊 Уникальные значения:
Уникальных валют: 1009
Уникальных единиц измерения: 129
Уникальных символов: 206
Уникальных коротких названий (shortName): 35

🏆 Топ-5 единиц измерения (unit) по числу записей:
unit
доллар США         361
евро               270
фунт стерлингов     70
датская крона       31
шведская крона      30
Name: count, dtype: int64

💱 Уникальные валюты в датасете:
  • иракский динар
  • кувейтский динар
  • бангладешская така
  • египетский фунт
  • афгани
  • аргентинское песо
  • алжирский динар
  • таджикский сомони
  • чилийское песо
  • лаосский кип
  • марокканский дирхам
  • дирхам ОАЭ
  • ангольская кванза
  • боливиано
  • мозамбикский метикал
  • Кина
  • бахрейнский динар
  • кьят
  • суринамский доллар
  • южнокорейская вона
  • кенийский шиллинг
  • гамбийский даласи
  • непальская рупия
  • нигерийская найра
  • гаитянский гурд
  • перуанский новый соль
  • камбоджийский риель
  • эфиопский быр
  • брунейский доллар
  • мальд

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨